# P01：描述性统计、可视化与回归分析| 项目 | 内容 ||------|------|| 课程 | 数据分析与经济决策（ds2026） || 姓名 | 柯骋 || 学号 | 25210150 || 日期 | 2026-05-21 |

## 1. 加载清洗后数据

In [ ]:
import osimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlibimport seaborn as snsfrom scipy import statsimport statsmodels.api as smimport warningswarnings.filterwarnings('ignore')# 中文字体设置matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']matplotlib.rcParams['axes.unicode_minus'] = Falseplt.style.use('seaborn-v0_8-whitegrid')BASE_DIR = os.path.dirname(os.path.abspath("__file__"))STOCKS = [    {"code": "600036", "name": "招商银行", "industry": "银行"},    {"code": "601398", "name": "工商银行", "industry": "银行"},    {"code": "002594", "name": "比亚迪",   "industry": "汽车"},    {"code": "601633", "name": "长城汽车", "industry": "汽车"},    {"code": "000002", "name": "万科A",    "industry": "房地产"},    {"code": "600048", "name": "保利发展", "industry": "房地产"},    {"code": "600519", "name": "贵州茅台", "industry": "白酒"},    {"code": "000858", "name": "五粮液",   "industry": "白酒"},    {"code": "601857", "name": "中国石油", "industry": "能源"},    {"code": "000063", "name": "中兴通讯", "industry": "通讯"},]INDUSTRY_COLORS = {    "银行": "#e74c3c", "汽车": "#3498db", "房地产": "#2ecc71",    "白酒": "#f39c12", "能源": "#9b59b6", "通讯": "#1abc9c"}# 加载清洗后数据data = pd.read_csv(os.path.join(BASE_DIR, "data", "clean", "stock_clean.csv"))data["日期"] = pd.to_datetime(data["日期"])print(f"数据加载完成: {data.shape}")

## 2. 基本统计量

In [ ]:
# 计算日收益率的描述性统计trading_days = 252stats_table = []for stock in STOCKS:    code = stock["code"]    name = stock["name"]    industry = stock["industry"]        stock_data = data[data["code"] == code].copy()    stock_data = stock_data.sort_values("日期")    returns = stock_data["日收益率"].dropna()        if len(returns) == 0:        continue        # 年化均值和波动率    annual_mean = returns.mean() * trading_days * 100    annual_vol = returns.std() * np.sqrt(trading_days) * 100    skewness = returns.skew()    kurtosis = returns.kurtosis()        # 最大回撤    cum_return = (1 + returns).cumprod()    running_max = cum_return.cummax()    drawdown = (cum_return - running_max) / running_max    max_drawdown = drawdown.min() * 100        stats_table.append({        "股票": f"{name}", "行业": industry,        "年化均值(%)": f"{annual_mean:.2f}",        "年化波动率(%)": f"{annual_vol:.2f}",        "偏度": f"{skewness:.3f}",        "峰度": f"{kurtosis:.3f}",        "最大回撤(%)": f"{max_drawdown:.2f}"    })stats_df = pd.DataFrame(stats_table)stats_df

## 3. 可视化

### 图1：归一化收盘价走势图

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))for stock in STOCKS:    code = stock["code"]    name = stock["name"]    industry = stock["industry"]        stock_data = data[data["code"] == code].sort_values("日期")    if len(stock_data) == 0:        continue        # 归一化：以2020-01-01为基准    first_close = stock_data["收盘价"].iloc[0]    normalized = stock_data["收盘价"] / first_close        color = INDUSTRY_COLORS.get(industry, "gray")    ax.plot(stock_data["日期"], normalized, label=f"{name}({industry})",             color=color, alpha=0.8, linewidth=1.2)# 叠加沪深300hs300 = data[data["code"] == "600036"][["日期", "hs300_close"]].drop_duplicates().sort_values("日期")if len(hs300) > 0:    hs300_norm = hs300["hs300_close"] / hs300["hs300_close"].iloc[0]    ax.plot(hs300["日期"], hs300_norm, label="沪深300",             color="black", linewidth=2, linestyle="--", alpha=0.7)ax.set_title("图1：10只股票归一化收盘价走势（2020-01-01 = 1）", fontsize=14)ax.set_xlabel("日期", fontsize=12)ax.set_ylabel("归一化收盘价", fontsize=12)ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig(os.path.join(BASE_DIR, "output", "fig1_normalized_price.png"), dpi=150, bbox_inches="tight")plt.show()print("图1已保存")

In [ ]:
**图1解读**：从归一化走势可以看出，2020-2026年间比亚迪涨幅最为显著，体现了新能源汽车行业的爆发式增长。白酒板块（贵州茅台、五粮液）在2021年初达到高点后经历了显著回调。房地产板块（万科A、保利发展）整体表现低迷，反映了行业下行周期。银行板块走势相对稳健但涨幅有限，呈现典型的防御性特征。

### 图2：日收益率分布图

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))axes = axes.flatten()for i, stock in enumerate(STOCKS):    code = stock["code"]    name = stock["name"]    industry = stock["industry"]        stock_data = data[data["code"] == code]    returns = stock_data["日收益率"].dropna()        if len(returns) == 0:        continue        ax = axes[i]    color = INDUSTRY_COLORS.get(industry, "gray")        # 直方图    ax.hist(returns, bins=50, density=True, alpha=0.6, color=color, edgecolor="white")        # 叠加正态分布曲线    x = np.linspace(returns.min(), returns.max(), 100)    ax.plot(x, stats.norm.pdf(x, returns.mean(), returns.std()),             'k-', linewidth=1.5, alpha=0.8)        mean_val = returns.mean() * 100    std_val = returns.std() * 100    ax.set_title(f"{name}({industry})\nμ={mean_val:.3f}%, σ={std_val:.3f}%", fontsize=9)    ax.set_xlabel("日收益率", fontsize=8)    ax.set_ylabel("密度", fontsize=8)plt.suptitle("图2：10只股票日收益率分布（叠加正态分布曲线）", fontsize=14, y=1.02)plt.tight_layout()plt.savefig(os.path.join(BASE_DIR, "output", "fig2_return_distribution.png"), dpi=150, bbox_inches="tight")plt.show()print("图2已保存")

In [ ]:
**图2解读**：所有股票的日收益率分布均呈现"尖峰厚尾"特征，即峰度高于正态分布，极端值出现频率高于正态假设。比亚迪和长城汽车的波动率最大，符合成长型汽车股的特征；银行股（招商银行、工商银行）波动率最低，体现防御属性。大多数股票收益率分布略呈左偏，说明出现极端负收益的概率高于极端正收益。

### 图3：收益率相关系数热力图

In [ ]:
# 计算相关系数矩阵returns_wide = pd.DataFrame()for stock in STOCKS:    code = stock["code"]    stock_data = data[data["code"] == code][["日期", "日收益率"]].dropna()    stock_data = stock_data.set_index("日期")    returns_wide[stock["name"]] = stock_data["日收益率"]# 按行业排序industry_order = ["银行", "汽车", "房地产", "白酒", "能源", "通讯"]sorted_stocks = sorted(STOCKS, key=lambda x: industry_order.index(x["industry"]))sorted_names = [s["name"] for s in sorted_stocks]corr_matrix = returns_wide[sorted_names].corr()fig, ax = plt.subplots(figsize=(10, 8))sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,            vmin=-0.5, vmax=1, ax=ax, square=True,            linewidths=0.5, cbar_kws={"shrink": 0.8})ax.set_title("图3：10只股票日收益率相关系数热力图", fontsize=14)plt.tight_layout()plt.savefig(os.path.join(BASE_DIR, "output", "fig3_correlation_heatmap.png"), dpi=150, bbox_inches="tight")plt.show()print("图3已保存")

In [ ]:
**图3解读**：同行业内股票的相关性明显高于跨行业股票。招商银行与工商银行的相关系数最高，同为银行股，受相同宏观因素驱动。比亚迪与长城汽车也有较高的正相关性。白酒板块内部（茅台与五粮液）相关性同样显著。跨行业方面，银行与房地产相关度较高，反映了金融与地产的关联性。而通讯（中兴通讯）与其他板块的相关性最低，体现了行业独立性。

### 图4：宏观指标与股市关系

In [ ]:
# CPI与沪深300月度收益率的散点图# 计算沪深300月度收益率hs300_data = pd.read_csv(os.path.join(BASE_DIR, "data", "index", "index_000300.csv"))hs300_data["日期"] = pd.to_datetime(hs300_data["日期"])hs300_data = hs300_data.set_index("日期").sort_index()hs300_monthly = hs300_data["收盘价"].resample("M").last()hs300_monthly_return = np.log(hs300_monthly / hs300_monthly.shift(1)).dropna() * 100hs300_monthly_return.name = "hs300_return"cpi = pd.read_csv(os.path.join(BASE_DIR, "data", "macro", "macro_cpi.csv"))cpi["日期"] = pd.to_datetime(cpi["日期"])cpi = cpi.set_index("日期").sort_index()# 合并merged = pd.DataFrame({"hs300_return": hs300_monthly_return, "CPI同比": cpi["CPI同比"]}).dropna()fig, ax = plt.subplots(figsize=(10, 6))ax.scatter(merged["CPI同比"], merged["hs300_return"], alpha=0.6, color="#3498db", s=60, edgecolor="white")# 线性拟合slope, intercept, r_value, p_value, std_err = stats.linregress(merged["CPI同比"], merged["hs300_return"])x_fit = np.linspace(merged["CPI同比"].min(), merged["CPI同比"].max(), 100)y_fit = slope * x_fit + interceptax.plot(x_fit, y_fit, "r-", linewidth=2, alpha=0.8)ax.set_title("图4：CPI同比增速与沪深300月度收益率", fontsize=14)ax.set_xlabel("CPI同比增速(%)", fontsize=12)ax.set_ylabel("沪深300月度收益率(%)", fontsize=12)# 标注Pearson相关系数pearson_r = merged["CPI同比"].corr(merged["hs300_return"])ax.text(0.05, 0.95, f"Pearson r = {pearson_r:.3f}\nslope = {slope:.3f}\np-value = {p_value:.4f}",         transform=ax.transAxes, fontsize=11, verticalalignment="top",        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))plt.tight_layout()plt.savefig(os.path.join(BASE_DIR, "output", "fig4_cpi_stock_scatter.png"), dpi=150, bbox_inches="tight")plt.show()print("图4已保存")

In [ ]:
**图4解读**：CPI与沪深300收益率的关系需要结合通胀区间具体分析，低通胀阶段可能正相关（经济回暖），高通胀阶段可能负相关（货币收紧预期)。

### 图5（选做）：财务指标跨公司对比

In [ ]:
# ROE跨公司对比finance = pd.read_csv(os.path.join(BASE_DIR, "data", "finance", "finance_ratios.csv"))roe_data = finance[finance["indicator"] == "ROE"].copy()fig, ax = plt.subplots(figsize=(12, 6))for stock in STOCKS:    code = stock["code"]    name = stock["name"]    industry = stock["industry"]        stock_roe = roe_data[roe_data["code"] == code].sort_values("year")    if len(stock_roe) == 0:        continue        color = INDUSTRY_COLORS.get(industry, "gray")    ax.plot(stock_roe["year"], stock_roe["value"],             marker="o", label=f"{name}({industry})", color=color, linewidth=1.5, markersize=4)ax.set_title("图5：10只股票近5年ROE对比", fontsize=14)ax.set_xlabel("年度", fontsize=12)ax.set_ylabel("ROE(%)", fontsize=12)ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig(os.path.join(BASE_DIR, "output", "fig5_roe_comparison.png"), dpi=150, bbox_inches="tight")plt.show()print("图5已保存")

In [ ]:
**图5解读**：白酒板块（贵州茅台、五粮液）的ROE长期领先于其他行业，体现了白酒行业轻资产、高盈利的商业模式优势。银行股ROE相对稳定但近年来呈下行趋势，受净息差收窄影响。房地产行业ROE下降最为明显，万科A和保利发展均受行业景气下行拖累。比亚迪ROE近年快速上升，反映了新能源汽车行业的高景气度和规模效应释放。

## 4. CAPM模型估计

In [ ]:
# CAPM: r_i - r_f = alpha + beta * (r_m - r_f) + epsilonrf_daily = 0.02 / 252  # 年化2%的无风险利率，日频换算# 沪深300收益率作为市场收益率hs300_returns = data[data["code"] == "600036"][["日期", "日收益率"]].dropna()hs300_returns = hs300_returns.set_index("日期")["日收益率"]hs300_returns.name = "market_return"capm_results = []for stock in STOCKS:    code = stock["code"]    name = stock["name"]    industry = stock["industry"]        stock_data = data[data["code"] == code][["日期", "日收益率"]].dropna()    stock_data = stock_data.set_index("日期")["日收益率"]    stock_data.name = "stock_return"        # 合并    merged = pd.merge(stock_data, hs300_returns, left_index=True, right_index=True, how="inner")        if len(merged) < 100:        continue        # 超额收益    merged["excess_stock"] = merged["stock_return"] - rf_daily    merged["excess_market"] = merged["market_return"] - rf_daily        # OLS回归    X = sm.add_constant(merged["excess_market"])    y = merged["excess_stock"]    model = sm.OLS(y, X).fit()        alpha = model.params["const"]    alpha_pval = model.pvalues["const"]    beta = model.params["excess_market"]    beta_ci = model.conf_int().loc["excess_market"]    r_squared = model.rsquared        capm_results.append({        "股票": name, "行业": industry,        "α̂": f"{alpha:.6f}", "p值(α)": f"{alpha_pval:.4f}",        "β̂": f"{beta:.4f}", "95%CI(β)": f"[{beta_ci[0]:.4f}, {beta_ci[1]:.4f}]",        "R²": f"{r_squared:.4f}",        "alpha_raw": alpha, "beta_raw": beta,        "alpha_pval_raw": alpha_pval, "r2_raw": r_squared,        "beta_ci_low": beta_ci[0], "beta_ci_high": beta_ci[1]    })capm_df = pd.DataFrame(capm_results)capm_display = capm_df[["股票", "行业", "α̂", "p值(α)", "β̂", "95%CI(β)", "R²"]]capm_display

### CAPM Beta系数点图

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))for _, row in capm_df.iterrows():    industry = row["行业"]    color = INDUSTRY_COLORS.get(industry, "gray")        ax.errorbar(row["beta_raw"], row["股票"],                 xerr=[[row["beta_raw"] - row["beta_ci_low"]], [row["beta_ci_high"] - row["beta_raw"]]],                fmt="o", color=color, markersize=8, capsize=5, linewidth=1.5,                label=industry if industry not in [r["行业"] for r in capm_results[:capm_df.index.get_loc(_)]] else None)# 参考线 beta=1ax.axvline(x=1.0, color="red", linestyle="--", linewidth=1.5, alpha=0.7, label="β=1")ax.set_title("CAPM Beta系数点图（95%置信区间）", fontsize=14)ax.set_xlabel("Beta系数", fontsize=12)ax.set_ylabel("股票", fontsize=12)# 去除重复图例handles, labels = ax.get_legend_handles_labels()by_label = dict(zip(labels, handles))ax.legend(by_label.values(), by_label.keys(), fontsize=9)plt.tight_layout()plt.savefig(os.path.join(BASE_DIR, "output", "fig6_capm_beta.png"), dpi=150, bbox_inches="tight")plt.show()print("Beta图已保存")

### CAPM分析讨论

In [ ]:
# 讨论1：哪些股票 β̂ > 1？high_beta = capm_df[capm_df["beta_raw"] > 1]print("β̂ > 1 的股票（周期性股票）：")for _, row in high_beta.iterrows():    print(f"  {row['股票']}({row['行业']}): β={row['beta_raw']:.4f}")print(f"\n这些股票属于周期性行业（汽车、房地产、白酒等），其股价波动大于市场平均水平。")print("这与'周期性 vs 防御性'行业分类吻合：")print("  - 周期性行业（汽车、房地产）在经济上行期弹性更大，β > 1")print("  - 防御性行业（银行、能源）波动相对较小，β < 1")

In [ ]:
# 讨论2：Alpha是否显著异于零？sig_alpha = capm_df[capm_df["alpha_pval_raw"] < 0.05]print(f"Alpha显著异于零的股票（p < 0.05）：{len(sig_alpha)} 只")for _, row in sig_alpha.iterrows():    print(f"  {row['股票']}: α={row['alpha_raw']:.6f}, p={row['alpha_pval_raw']:.4f}")print(f"\nAlpha显著意味着：在控制了市场风险后，该股票仍存在超额收益（或亏损）。")print("正的显著Alpha表示股票获得了超过CAPM预测的回报，可能源于选股能力或因子暴露。")print("负的显著Alpha表示股票回报不及CAPM预测，可能是特定风险未被模型捕捉。")

In [ ]:
# 讨论3：R²最高和最低的股票max_r2 = capm_df.loc[capm_df["r2_raw"].idxmax()]min_r2 = capm_df.loc[capm_df["r2_raw"].idxmin()]print(f"R²最高：{max_r2['股票']}({max_r2['行业']})，R²={max_r2['r2_raw']:.4f}")print(f"R²最低：{min_r2['股票']}({min_r2['行业']})，R²={min_r2['r2_raw']:.4f}")print(f"\n解释：")print(f"  R²衡量的是市场收益率对个股收益率变动的解释比例。")print(f"  {max_r2['股票']}的R²最高，说明其走势与大盘高度一致，系统性风险是主要风险来源。")print(f"  {min_r2['股票']}的R²最低，说明其受公司特定因素影响更大，")print(f"  个股特质风险（idiosyncratic risk）占比较大，大盘走势对其解释力有限。")

## 5. 宏观指标对股票收益率的影响（选做）

In [ ]:
# M2增速对月度收益率的影响m2 = pd.read_csv(os.path.join(BASE_DIR, "data", "macro", "macro_m2.csv"))m2["日期"] = pd.to_datetime(m2["日期"])# 计算各股票月度收益率monthly_results = []for stock in STOCKS:    code = stock["code"]    name = stock["name"]    industry = stock["industry"]        stock_data = data[data["code"] == code].copy().set_index("日期").sort_index()    monthly_close = stock_data["收盘价"].resample("M").last()    monthly_return = np.log(monthly_close / monthly_close.shift(1)).dropna() * 100    monthly_return.name = "monthly_return"        # 合并M2    m2_monthly = m2.set_index("日期")["M2同比"]    merged = pd.DataFrame({"return": monthly_return, "M2": m2_monthly}).dropna()        if len(merged) < 20:        continue        X = sm.add_constant(merged["M2"])    y = merged["return"]    model = sm.OLS(y, X).fit()        monthly_results.append({        "股票": name, "行业": industry,        "γ̂": f"{model.params['M2']:.4f}",        "p值": f"{model.pvalues['M2']:.4f}",        "gamma_raw": model.params["M2"],        "pval_raw": model.pvalues["M2"]    })monthly_df = pd.DataFrame(monthly_results)monthly_df

In [ ]:
# 可视化M2敏感度fig, ax = plt.subplots(figsize=(10, 6))for _, row in monthly_df.iterrows():    industry = row["行业"]    color = INDUSTRY_COLORS.get(industry, "gray")    marker = "o" if row["pval_raw"] < 0.05 else "x"    ax.scatter(row["gamma_raw"], row["股票"], color=color, s=100, marker=marker, zorder=5)ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)ax.set_title("M2同比增速对月度收益率的敏感度（γ̂）", fontsize=14)ax.set_xlabel("γ̂（M2敏感度）", fontsize=12)ax.set_ylabel("股票", fontsize=12)ax.annotate("o = p<0.05显著  x = 不显著", xy=(0.02, 0.02), xycoords="axes fraction", fontsize=10)plt.tight_layout()plt.savefig(os.path.join(BASE_DIR, "output", "fig7_m2_sensitivity.png"), dpi=150, bbox_inches="tight")plt.show()print("M2敏感度图已保存")# 讨论print("\n讨论：")print("M2增速代表市场流动性水平。流动性充裕时，资金更多流入股市推升估值。")print("不同行业对流动性的敏感度不同：")print("  - 高贝塔行业（汽车、白酒）对流动性更敏感，M2扩张时涨幅更大")print("  - 防御性行业（银行、能源）对流动性敏感度较低，走势更依赖基本面")

## 6. 总结

In [ ]:
print('''========== 作业总结 ==========1. 数据获取：成功获取10只A股股票5年后复权行情、2个指数、2项宏观指标、10只股票财务数据2. 数据清洗：完成缺失值填充、日期格式统一、类型检查、重复值处理、离群值标注6个步骤3. 数据存储：CSV（方式A）+ SQLite（方式C），支持跨表SQL查询4. 可视化：完成5张图表（归一化走势、收益率分布、相关热力图、CPI散点图、ROE对比）5. CAPM回归：估计10只股票的Alpha和Beta，完成3个讨论问题6. 宏观分析：M2增速对月度收益率的影响分析主要发现：- 新能源汽车（比亚迪）是2020-2026年涨幅最大的板块- 白酒行业ROE长期领先，但2021年后经历显著回调- 同行业股票相关性显著高于跨行业- 周期性行业Beta > 1，防御性行业Beta < 1- M2流动性对高贝塔行业影响更显著==============================''')